# ClinHallu: Whisper large-v3 / turbo evaluation (Colab T4)
Budget: about 40 min of T4 time (≈1.2 compute units). Runtime → Change runtime type → **T4 GPU**.
The notebook rebuilds the pilot split deterministically (same seeds as the local run), then evaluates the large models.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/hamzak-27/whisper-clinical-hallucination.git
%cd whisper-clinical-hallucination
!pip -q install faster-whisper jiwer kokoro-onnx soundfile

In [ ]:
%%bash
mkdir -p models/kokoro data/raw/esc50
wget -q -O models/kokoro/kokoro-v1.0.onnx https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx
wget -q -O models/kokoro/voices-v1.0.bin https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin
wget -q -O /tmp/esc50.zip https://github.com/karolpiczak/ESC-50/archive/master.zip && unzip -q /tmp/esc50.zip -d data/raw/esc50
ls models/kokoro data/raw/esc50

In [ ]:
# Build the pilot split (CPU TTS, ~3-5 min). Same seeds/params as the local run.
!cd src && python build_audio.py --split pilot --n 4 --gaps 1,12 2>&1 | tail -1

In [ ]:
# Evaluate. Order = most important first, so a budget cut-off still leaves useful results.
RUNS = [("large-v3","default"), ("large-v3","no_prev"), ("large-v3-turbo","default"), ("large-v3-turbo","no_prev")]
import subprocess
for m,p in RUNS:
    print(m,p, flush=True)
    subprocess.run(f"cd src && python run_asr.py --split pilot --model {m} --preset {p} --device cuda --compute float16", shell=True)

In [ ]:
!cd src && python aggregate.py pilot | head -40

In [ ]:
# Save results: download the zip (results are small JSONL).
!zip -qr results_colab.zip results
from google.colab import files; files.download('results_colab.zip')